# 01 — Data mining

**Purpose.** Mine git history from a curated set of popular OSS repos to produce SFT training trajectories for the Graft agent. Every signal here comes from real human commits — no frontier-model calls.

**Inputs.** `TARGET_REPOS` (a list of clone URLs) and a `CLONE_DIR` workspace.

**Outputs.** `OUTPUT_PATH` — a JSONL file where every line is a verified trajectory:

```json
{"messages": [...], "reward": 1.0, "meta": {"repo": "...", "dep": "...", "from_version": "...", "to_version": "...", "commit": "..."}}
```

**Pipeline.**
1. Clone or update each repo.
2. Walk commits; identify ones that change a manifest **and** application code (no test/CI/docs-only commits).
3. Verify: install deps and run the test suite at the bump commit. Discard if exit code != 0.
4. Convert the surviving diffs into a sequence of Graft tool calls: `read_changelog` → one `edit_file` per non-test source file → `run_tests` → `submit`.
5. Save.

In [ ]:
# ---------- CONFIGURATION (user-editable) ----------
from pathlib import Path

TARGET_REPOS = [
    "https://github.com/django/django",
    "https://github.com/pallets/flask",
    "https://github.com/tiangolo/fastapi",
    "https://github.com/psf/requests",
    "https://github.com/numpy/numpy",
    "https://github.com/pydantic/pydantic",
    "https://github.com/sqlalchemy/sqlalchemy",
    "https://github.com/celery/celery",
    "https://github.com/encode/httpx",
    "https://github.com/pytest-dev/pytest",
]

CLONE_DIR = Path("./repos")
OUTPUT_PATH = Path("./data/trajectories.jsonl")
MAX_COMMITS_PER_REPO = 2000
MIN_TEST_FILES = 3
VERIFY_TEST_TIMEOUT = 300        # seconds; per-commit hard cap
MAX_TRAJECTORIES_PER_REPO = 50   # cap to keep output manageable

In [ ]:
# ---------- imports ----------
import difflib
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import uuid
from collections import Counter
from dataclasses import dataclass

import git
import httpx
from packaging.version import InvalidVersion, Version

MANIFEST_FILES = {
    "requirements.txt", "requirements-dev.txt", "requirements-test.txt",
    "setup.cfg", "pyproject.toml", "Pipfile", "Pipfile.lock",
    "package.json", "package-lock.json", "yarn.lock",
    "Cargo.toml", "Cargo.lock",
}

TEST_DIRS = ("test", "tests", "spec", "__tests__")
DOC_PARTS = (".github", ".circleci", "docs/", ".md", ".rst", ".txt")

SYSTEM_PROMPT = """You are Graft, an autonomous coding agent that upgrades a single dependency in a project from one version to another and keeps the project's test suite passing.

Workflow:
1. read_changelog
2. grep_repo / ast_query / read_file to locate call sites
3. edit_file to apply minimal patches (NEVER touch test files or test config)
4. run_tests
5. submit"""

In [ ]:
# ---------- Step 1: clone / update repos ----------
CLONE_DIR.mkdir(parents=True, exist_ok=True)

def clone_or_update(url: str) -> Path:
    repo_name = url.rstrip("/").split("/")[-1]
    dest = CLONE_DIR / repo_name
    if dest.exists():
        try:
            git.Repo(dest).remotes.origin.pull()
            print(f"  pulled  {repo_name}")
        except Exception as e:
            print(f"  pull failed {repo_name}: {e}")
    else:
        print(f"  cloning {repo_name} ...")
        git.Repo.clone_from(url, dest)
        print(f"  cloned  {repo_name}")
    return dest

repo_paths = []
for url in TARGET_REPOS:
    try:
        repo_paths.append(clone_or_update(url))
    except Exception as e:
        print(f"  SKIP {url}: {e}")
print(f"Total repos available: {len(repo_paths)}")

In [ ]:
# ---------- Step 2: candidate identification ----------
def _is_test_path(p: str) -> bool:
    parts = p.replace("\\", "/").split("/")
    return any(seg in TEST_DIRS for seg in parts) or \
        any(part in {"conftest.py", "pytest.ini", "jest.config.js"} for part in parts)

def _is_doc_or_ci(p: str) -> bool:
    pl = p.lower()
    return any(pl.startswith(x) or pl.endswith(x) for x in DOC_PARTS)

def is_candidate(commit: git.Commit, parent: git.Commit) -> bool:
    try:
        diff = parent.diff(commit)
    except Exception:
        return False
    changed = [d.b_path for d in diff if d.b_path]
    if not changed:
        return False
    has_manifest_change = any(os.path.basename(f) in MANIFEST_FILES for f in changed)
    has_app_code_change = any(
        f.endswith((".py", ".js", ".ts", ".rs")) and not _is_test_path(f) and not _is_doc_or_ci(f)
        for f in changed
    )
    only_meta = all(
        _is_test_path(f) or _is_doc_or_ci(f) or os.path.basename(f) in MANIFEST_FILES
        for f in changed
    )
    return has_manifest_change and has_app_code_change and not only_meta

def collect_candidates(repo_path: Path):
    repo = git.Repo(repo_path)
    out = []
    head = repo.head.commit
    visited = 0
    for commit in repo.iter_commits(head, max_count=MAX_COMMITS_PER_REPO):
        if not commit.parents:
            continue
        parent = commit.parents[0]
        try:
            if is_candidate(commit, parent):
                out.append((parent, commit))
        except Exception:
            pass
        visited += 1
    return out

all_candidates = {}
for rp in repo_paths:
    cands = collect_candidates(rp)
    if len(cands) >= 1:
        all_candidates[rp] = cands
        print(f"  {rp.name}: {len(cands)} candidate commits")
print(f"Total candidates: {sum(len(v) for v in all_candidates.values())}")

In [ ]:
# ---------- Step 3: verify candidates by running their test suite ----------
# This is expensive. Each verification clones a worktree and runs pytest in a fresh venv.
# We discard candidates that don't leave the test suite green at the bump commit.

def _detect_install_cmd(workdir: Path) -> str:
    if (workdir / "pyproject.toml").exists():
        return "pip install -q -e .[test] 2>/dev/null || pip install -q -e . 2>/dev/null || true"
    if (workdir / "setup.py").exists():
        return "pip install -q -e . 2>/dev/null || true"
    if (workdir / "requirements.txt").exists():
        return "pip install -q -r requirements.txt 2>/dev/null || true"
    return "true"

def verify_at_commit(repo_path: Path, commit: git.Commit, timeout: int = VERIFY_TEST_TIMEOUT) -> tuple[bool, int]:
    """Returns (passed_cleanly, baseline_passed_count)."""
    repo = git.Repo(repo_path)
    wt_dir = Path(tempfile.mkdtemp(prefix="graft-wt-"))
    try:
        try:
            repo.git.worktree("add", "--detach", str(wt_dir), commit.hexsha)
        except git.GitCommandError as e:
            return False, 0
        install = _detect_install_cmd(wt_dir)
        cmd = f"python -m venv .venv && . .venv/bin/activate && pip install -q --upgrade pip && {install} && pip install -q pytest && pytest -q --no-header --tb=no"
        try:
            proc = subprocess.run(
                cmd, shell=True, cwd=str(wt_dir), capture_output=True, text=True,
                timeout=timeout, executable="/bin/bash" if os.name != "nt" else None,
            )
            text = (proc.stdout or "") + (proc.stderr or "")
            m = re.search(r"(\d+)\s+passed", text)
            passed_count = int(m.group(1)) if m else 0
            return proc.returncode == 0 and passed_count > 0, passed_count
        except subprocess.TimeoutExpired:
            return False, 0
    finally:
        try:
            repo.git.worktree("remove", "--force", str(wt_dir))
        except Exception:
            pass
        shutil.rmtree(wt_dir, ignore_errors=True)

# NOTE: this verification step is heavy. Set RUN_VERIFICATION=False during a dry run.
RUN_VERIFICATION = False  # flip to True for the real mining run
verified = {}
if RUN_VERIFICATION:
    for rp, cands in all_candidates.items():
        kept = []
        for parent, bump in cands[:MAX_TRAJECTORIES_PER_REPO]:
            ok, baseline = verify_at_commit(rp, bump)
            if ok:
                kept.append((parent, bump, baseline))
        if kept:
            verified[rp] = kept
            print(f"  {rp.name}: {len(kept)} verified")
else:
    # Skip verification — assume each candidate is OK with a placeholder baseline.
    for rp, cands in all_candidates.items():
        verified[rp] = [(p, b, 0) for p, b in cands[:MAX_TRAJECTORIES_PER_REPO]]
    print("Verification skipped (dry-run mode). Set RUN_VERIFICATION=True for the real pipeline.")
print(f"Total verified: {sum(len(v) for v in verified.values())}")

In [ ]:
# ---------- Step 4: helpers for trajectory reconstruction ----------
def extract_minimal_diff(old: str, new: str) -> tuple[str, str] | tuple[None, None]:
    """Return the smallest contiguous changed block as (old_str, new_str).

    Walks unified_diff hunks and picks the first one where the old portion is unique in `old`.
    Returns (None, None) if no usable hunk is found.
    """
    if old == new:
        return None, None
    matcher = difflib.SequenceMatcher(a=old, b=new, autojunk=False)
    blocks = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        old_str = old[max(0, i1-40):min(len(old), i2+40)]
        new_str = new[max(0, j1-40):min(len(new), j2+40)]
        if old_str and old.count(old_str) == 1:
            return old_str, new_str
        blocks.append((tag, i1, i2, j1, j2))
    # Fallback: small token windows
    for tag, i1, i2, j1, j2 in blocks:
        if tag in ("replace", "delete"):
            old_chunk = old[i1:i2]
            new_chunk = new[j1:j2] if tag == "replace" else ""
            if old_chunk and old.count(old_chunk) == 1:
                return old_chunk, new_chunk
    return None, None

def detect_dep_change(parent: git.Commit, bump: git.Commit) -> tuple[str, str, str] | None:
    """Inspect manifest diffs to extract (dep_name, from_version, to_version).

    Heuristic: scan changed manifest files for one-line version bumps in formats:
      - requirements: 'pkg==1.2.3'
      - pyproject:   '"pkg>=1.2,<2"'
      - package.json: '"pkg": "1.2.3"'
    Returns None if it can't pin one specific dep.
    """
    diff = parent.diff(bump, create_patch=True)
    candidates = []
    for d in diff:
        if d.b_path and os.path.basename(d.b_path) in MANIFEST_FILES:
            try:
                patch = d.diff.decode("utf-8", errors="replace")
            except Exception:
                continue
            removed = [ln[1:] for ln in patch.splitlines() if ln.startswith("-") and not ln.startswith("---")]
            added = [ln[1:] for ln in patch.splitlines() if ln.startswith("+") and not ln.startswith("+++")]
            for r, a in zip(removed, added):
                m_r = re.search(r"([A-Za-z0-9_\-]+)[><=~^]+([0-9][0-9A-Za-z.\-]*)", r)
                m_a = re.search(r"([A-Za-z0-9_\-]+)[><=~^]+([0-9][0-9A-Za-z.\-]*)", a)
                if m_r and m_a and m_r.group(1) == m_a.group(1) and m_r.group(2) != m_a.group(2):
                    candidates.append((m_r.group(1), m_r.group(2), m_a.group(2)))
                    continue
                m_json_r = re.search(r'"([A-Za-z0-9_@/\-]+)"\s*:\s*"\^?~?([0-9][0-9A-Za-z.\-]*)"', r)
                m_json_a = re.search(r'"([A-Za-z0-9_@/\-]+)"\s*:\s*"\^?~?([0-9][0-9A-Za-z.\-]*)"', a)
                if m_json_r and m_json_a and m_json_r.group(1) == m_json_a.group(1):
                    candidates.append((m_json_r.group(1), m_json_r.group(2), m_json_a.group(2)))
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        return None
    # Prefer the one with the largest version delta
    def _delta(t):
        try:
            return abs(int(Version(t[2]).release[0] if Version(t[2]).release else 0) - int(Version(t[1]).release[0] if Version(t[1]).release else 0))
        except (InvalidVersion, IndexError):
            return 0
    candidates.sort(key=_delta, reverse=True)
    return candidates[0]

def _make_tool_call(name: str, args: dict, idx: int) -> dict:
    return {"id": f"call_{idx:03d}", "name": name, "args": args}

CHANGELOG_CACHE: dict[str, str] = {}

def _fetch_changelog_text(dep: str, fv: str, tv: str) -> str:
    key = f"{dep}::{fv}::{tv}"
    if key in CHANGELOG_CACHE:
        return CHANGELOG_CACHE[key]
    text = f"Changelog for {dep} {fv} -> {tv} (synthesized; replace with live fetch when online)."
    try:
        with httpx.Client(timeout=5.0, headers={"User-Agent": "graft/0.1"}) as c:
            r = c.get(f"https://pypi.org/pypi/{dep}/json")
            if r.status_code == 200:
                releases = r.json().get("releases", {})
                if tv in releases and releases[tv]:
                    text = f"{dep} {tv} (from PyPI metadata): {len(releases[tv])} files released."
    except Exception:
        pass
    CHANGELOG_CACHE[key] = text
    return text

def _synthesise_result(tc: dict, dep: str, fv: str, tv: str, baseline: int) -> str:
    if tc["name"] == "read_changelog":
        return _fetch_changelog_text(dep, fv, tv)
    if tc["name"] == "edit_file":
        return f"Applied edit to {tc['args']['path']}"
    if tc["name"] == "run_tests":
        return json.dumps({"passed": baseline, "failed": 0, "errors": 0, "tracebacks": []})
    if tc["name"] == "submit":
        return "Episode complete; final evaluation pending."
    return ""

In [ ]:
# ---------- Step 4 (cont.): build a trajectory from each verified diff ----------
def build_trajectory(repo_path: Path, parent: git.Commit, bump: git.Commit, baseline: int) -> dict | None:
    dep_info = detect_dep_change(parent, bump)
    if dep_info is None:
        return None
    dep_name, fv, tv = dep_info

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Upgrade {dep_name} from {fv} to {tv}. "
                f"Test suite baseline: {baseline} passing, 0 failing."
            ),
        },
    ]

    tool_calls = [_make_tool_call("read_changelog", {"dep": dep_name, "from_version": fv, "to_version": tv}, 0)]

    diff = parent.diff(bump, create_patch=True)
    idx = 1
    for d in diff:
        path = d.b_path
        if not path or os.path.basename(path) in MANIFEST_FILES:
            continue
        if _is_test_path(path) or _is_doc_or_ci(path):
            continue
        try:
            old_blob = d.a_blob.data_stream.read().decode("utf-8", errors="replace") if d.a_blob else ""
            new_blob = d.b_blob.data_stream.read().decode("utf-8", errors="replace") if d.b_blob else ""
        except Exception:
            continue
        old_str, new_str = extract_minimal_diff(old_blob, new_blob)
        if old_str and new_str is not None and old_str != new_str:
            tool_calls.append(_make_tool_call("edit_file", {"path": path, "old_str": old_str, "new_str": new_str}, idx))
            idx += 1
        if idx >= 12:
            break

    if idx == 1:
        return None  # no usable edits

    tool_calls.append(_make_tool_call("run_tests", {}, idx)); idx += 1
    tool_calls.append(_make_tool_call("submit", {}, idx))

    for tc in tool_calls:
        messages.append({"role": "assistant", "content": None, "tool_calls": [tc]})
        messages.append({
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": _synthesise_result(tc, dep_name, fv, tv, baseline),
        })

    return {
        "messages": messages,
        "reward": 1.0,
        "meta": {
            "repo": str(repo_path),
            "dep": dep_name,
            "from_version": fv,
            "to_version": tv,
            "commit": bump.hexsha,
        },
    }

trajectories = []
lang_counts = Counter()
bump_kind_counts = Counter()

def _bump_kind(fv: str, tv: str) -> str:
    try:
        f, t = Version(fv), Version(tv)
    except InvalidVersion:
        return "unknown"
    if t.release[:1] != f.release[:1]:
        return "major"
    if len(t.release) > 1 and len(f.release) > 1 and t.release[:2] != f.release[:2]:
        return "minor"
    return "patch"

for rp, kept in verified.items():
    for parent, bump, baseline in kept:
        traj = build_trajectory(rp, parent, bump, baseline if baseline > 0 else 1)
        if traj is not None:
            trajectories.append(traj)
            lang_counts["python"] += 1
            bump_kind_counts[_bump_kind(traj["meta"]["from_version"], traj["meta"]["to_version"])] += 1

print(f"Trajectories built: {len(trajectories)}")
print(f"Languages: {dict(lang_counts)}")
print(f"Bump kinds: {dict(bump_kind_counts)}")

In [ ]:
# ---------- Step 5: write JSONL + summary ----------
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

total_candidates = sum(len(v) for v in all_candidates.values())
verified_count = sum(len(v) for v in verified.values())

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for traj in trajectories:
        f.write(json.dumps(traj) + "\n")

print("=" * 60)
print(f"Total candidates found  : {total_candidates}")
print(f"Verified (tests passed) : {verified_count}")
print(f"Trajectories written    : {len(trajectories)}")
print(f"Pass rate               : {(len(trajectories) / max(1, total_candidates)) * 100:.1f}%")
print(f"Language breakdown      : {dict(lang_counts)}")
print(f"Bump-type breakdown     : {dict(bump_kind_counts)}")
print(f"Output                  : {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size // 1024} KB)")